In [1]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 44.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 21.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 81.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#LXMERT con immagini e testo

installo i pacchetti necessari

In [3]:
!git clone https://github.com/huggingface/transformers

Cloning into 'transformers'...
remote: Enumerating objects: 146609, done.
remote: Counting objects: 100% (393/393), done.
remote: Compressing objects: 100% (256/256), done.
remote: Total 146609 (delta 181), reused 242 (delta 120), pack-reused 146216
Receiving objects: 100% (146609/146609), 151.08 MiB | 19.08 MiB/s, done.
Resolving deltas: 100% (108494/108494), done.


In [4]:
cd transformers

/content/transformers


In [5]:
ls

awesome-transformers.md  hubconf.py      README_hd.md       setup.py
CITATION.cff             ISSUES.md       README_ja.md       src/
CODE_OF_CONDUCT.md       LICENSE         README_ko.md       templates/
conftest.py              Makefile        README.md          tests/
CONTRIBUTING.md          model_cards/    README_zh-hans.md  utils/
docker/                  notebooks/      README_zh-hant.md
docs/                    pyproject.toml  scripts/
examples/                README_es.md    setup.cfg


In [6]:
cd examples/research_projects/lxmert

/content/transformers/examples/research_projects/lxmert


In [7]:
pip install wget

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9657 sha256=8e01ae86af81a25d8b9728697b38bdd47a391b14c44a249b009e4f3850eae2f2
  Stored in directory: /root/.cache/pip/wheels/8b/f1/7f/5c94f0a7a505ca1c81cd1d9208ae2064675d97582078e6c769
Successfully built wget


##Calcolo pseudo-log-likelihood

In [8]:
import os
import pickle
import torch
import numpy as np
from typing import Optional


In [15]:
class Pseudo_log_likelihood:

    def __init__(self, tokenizer, model: torch.nn.Module, model_name, sentence, features, normalized_boxes):
        self.tokenizer = tokenizer
        self.model = model
        self.model_name = model_name
        self.sentence = sentence
        self.features = features
        self.normalized_boxes = normalized_boxes
        self.cached_plls: Optional[np.ndarray] = None
        self.sent, self.score = self.pseudo_log_likelihood(self.sentence, self.model_name, self.features, self.normalized_boxes)


    def get_masked_seq(self, tokens, tokenized_words, tokenizer, model_name):
        """
        Masking out sentences word by word. If a word is tokenized into multiple subtokens, mask out subtokens linearly.
        i.e. for "hooligan":
        * we predict "ho"  knowing that 2 masks are still to come
        * we predict "##oli" with "ho" in context & knowing a mask is still to come
        * we predict "##gan" with "ho" and "##oli" in context
        """
        nr_masks = [len(elm) for elm in tokenized_words]

        masked_seq = []
        i = 1 #start with 0 because we want to predict for the first element
        j = 1
        while i < len(tokens) -1: #don't subtract 1 from len(tokens) to include the last element
            curr_masked_seq = []
            if nr_masks[j] == 1:
                curr_masked_seq = [tokens[ind] if ind != i else tokenizer.mask_token for ind in range(len(tokens))]
                masked_seq.append(curr_masked_seq)
                i += 1
                j += 1

            else:
                for k in range(nr_masks[j]):
                    curr_nr_masks = nr_masks[j] - k
                    curr_masked_seq = [tokens[ind] if (ind < i+k or ind >= i+k+curr_nr_masks) else tokenizer.mask_token for ind in range(len(tokens))]
                    masked_seq.append(curr_masked_seq)
                i += nr_masks[j]
                j += 1

        return masked_seq

    def prepare_input(self, sentence, model_name):

        tokens = self.tokenizer.tokenize(sentence)
        tokens = [self.tokenizer.cls_token] + tokens + [self.tokenizer.sep_token]
        #print(f"Tokens: {tokens}")
        tokenized_words = []

        for ind, tok in enumerate(tokens):
            if not re.match("#", tok):
                curr_word = [tok]
            else:
                curr_word.append(tok)

            #end a word
            if ind == len(tokens) - 1:
                tokenized_words.append(curr_word)
            else:
                if not re.match("#", tokens[ind+1]):
                    tokenized_words.append(curr_word)

        #print(f"Tokenized words: {tokenized_words}")

        masked_seq = self.get_masked_seq(tokens, tokenized_words, self.tokenizer, self.model_name)
        #print(f"Masked seq: {masked_seq}")

        return tokens, masked_seq

    def pseudo_log_likelihood(self, sentence, model_name, features, normalized_boxes):
        mask_token_id = self.tokenizer.mask_token_id
        max_len = 20

        """Come in VisualBERT, anche in LXMERT estraggo separatamente i token testuali e le feature visuali"""

        tokens, masked_seq = self.prepare_input(sentence, model_name)
        nr_tokens_to_predict = len(tokens) - 2 #because of CLS & SEP
        list_of_sents = [sentence] * nr_tokens_to_predict
        encoded_inputs = self.tokenizer(list_of_sents, padding='max_length', max_length=20)


        for i in range(len(masked_seq)):
            for j in range(len(masked_seq[i])):
                if masked_seq[i][j] == self.tokenizer.mask_token:
                    encoded_inputs["input_ids"][i][j] = self.tokenizer.mask_token_id

        input_ids = torch.tensor(encoded_inputs["input_ids"])
        attention_mask = torch.tensor(encoded_inputs["attention_mask"])

        #II torch.Size([7, 20]) AM torch.Size([7, 20]) TTI torch.Size([7, 20])
        print("II", input_ids.shape, "AM", attention_mask.shape)
        print("visual features", features.shape, "visual pos", normalized_boxes.shape)

        #features.shape = torch.Size([1, 36, 2048])
        #normalized_boxes = torch.Size([1, 36, 4])

        # calcola la pseudo-log-likelihood
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                visual_feats= torch.zeros(1, 36, 2048), #([1, 36, 2048])
                visual_attention_mask = torch.ones(features.shape[:-1], dtype=torch.long),
                visual_pos=torch.zeros(1, 36, 4) #torch.Size([1, 36, 4])
                )
        log_probs_fillers = []
        all_log_probs=[]
        predict_token = tokens[1:-1]
        logits = outputs.prediction_logits
        print("logits shape", logits.shape)


        for batch_elem, token, index in zip(range(len(logits)), predict_token, range(1, len(predict_token) + 1)): #no need for CLS & SEP
            all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])
            log_probs_fillers.append(all_log_probs[self.tokenizer.convert_tokens_to_ids(token)].item())
            #print(f"{self.tokenizer.convert_tokens_to_ids(token)} | {token} | {all_log_probs[self.tokenizer.convert_tokens_to_ids(token)].item()}\n")

        sentence_score = sum(log_probs_fillers)
        print(f" {sentence} | {sentence_score}\n")
        #self.cached_plls = np.array([sentence_score])

        return sentence, sentence_score

In [10]:
from IPython.display import clear_output, Image, display
import PIL.Image
import io
import json
import torch
import numpy as np
from processing_image import Preprocess
from visualizing_image import SingleImageViz
from modeling_frcnn import GeneralizedRCNN
from utils import Config
import utils
import wget
import cv2
from copy import deepcopy
from transformers import LxmertTokenizer, LxmertForPreTraining
import pandas as pd
import re
import os

##main

In [16]:
def main():

    force = True

    ## configurazione della rete R-CNN
    frcnn_cfg = Config.from_pretrained("unc-nlp/frcnn-vg-finetuned")
    frcnn = GeneralizedRCNN.from_pretrained("unc-nlp/frcnn-vg-finetuned", config=frcnn_cfg)
    image_preprocess = Preprocess(frcnn_cfg)

    result_plls = {}
    print(f"\nforce={force}\n")
    if os.path.exists("/content/"):
        data_dir = "/content/drive/MyDrive/Data"
    else:
        data_dir = "Data"
    img_folder= "/content/drive/MyDrive/Data/EventsRev_picture_stimuli/"
    models = {
        "LXMERT": (
            LxmertTokenizer.from_pretrained("unc-nlp/lxmert-base-uncased"),
            LxmertForPreTraining.from_pretrained("unc-nlp/lxmert-base-uncased")
        ),
    }
    filename = (f"{data_dir}/eventsRev_demo.csv")
    global_results = {}
    for model_name, (tokenizer, model) in list(models.items()):
        print(filename)
        results = {}
        global_results[filename] = results
        print("File Name:", filename.split("/")[-1])
        out_file_name = os.path.join(
            "/content/drive/MyDrive/Data/results", f"{model_name}_{os.path.basename(filename)}_sentence.txt"
        )
        data = pd.read_csv(filename, sep=";")
        results["len(data)"] = len(data)
        """save the results with the file name"""
        if force or (not os.path.exists(f"{filename}_result_plls.pkl")):
            for model_name, (tokenizer, model) in list(models.items()):
                model_results = {}
                results[model_name] = model_results
                result = {
                        "sentences": [],
                        "sent_ppls": [],
                        "num_words": [],
                        "num_tokens": [],
                        }
                print(f"\n\nEvaluating: {model_name}")

                """Per ogni frase associamo un'immagine sulla base del nome del file corrispondente"""
                for idx, row in enumerate(data.itertuples()):
                    sentence = row.Sentence
                    image_filename = row.file_name
                    image_path = os.path.join(img_folder, image_filename)

                    ## preprocessing dell'immagine
                    images, sizes, scales_yx = image_preprocess(image_path)
                    output_dict = frcnn(
                        images,
                        sizes,
                        scales_yx=scales_yx,
                        padding="max_detections",
                        max_detections=frcnn_cfg.max_detections,
                        return_tensors="pt",
                        )

                    # Normalizzazione delle boxes
                    normalized_boxes = output_dict.get("normalized_boxes")
                    features = output_dict.get("roi_features")

                    pll = Pseudo_log_likelihood(
                            tokenizer, model, model_name, sentence, features, normalized_boxes
                            )

                    #per ogni frase ci facciamo restituire il numero di parole e di token
                    tokens = tokenizer.tokenize(sentence)
                    num_tokens = len(tokens)
                    words = sentence.split()
                    num_words = len(words)
                    result["sentences"].append(pll.sent)
                    result["sent_ppls"].append(pll.score)
                    result["num_words"].append(num_words)
                    result["num_tokens"].append(num_tokens)

                # salviamo i risultati
                pd.DataFrame(result).to_csv(
                    out_file_name, sep="\t", header=None, index=None
                    )


if __name__ == '__main__':
    main()

loading configuration file cache
loading weights file https://cdn.huggingface.co/unc-nlp/frcnn-vg-finetuned/pytorch_model.bin from cache at /root/.cache/torch/transformers/57f6df6abe353be2773f2700159c65615babf39ab5b48114d2b49267672ae10f.77b59256a4cf8343ae0f923246a81489fc8d82f98d082edc2d2037c977c0d9d0
All model checkpoint weights were used when initializing GeneralizedRCNN.

All the weights of GeneralizedRCNN were initialized from the model checkpoint at unc-nlp/frcnn-vg-finetuned.
If your task is similar to the task the model of the checkpoint was trained on, you can already use GeneralizedRCNN for predictions without further training.

force=True

/content/drive/MyDrive/Data/eventsRev_demo.csv
File Name: eventsRev_demo.csv


Evaluating: LXMERT
II torch.Size([7, 20]) AM torch.Size([7, 20])
visual features torch.Size([1, 36, 2048]) visual pos torch.Size([1, 36, 4])
logits shape torch.Size([7, 20, 30522])
 The cop is arresting the criminal. | -39.808327466249466



<ipython-input-15-3cb0b6bfbc43>:117: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])


II torch.Size([7, 20]) AM torch.Size([7, 20])
visual features torch.Size([1, 36, 2048]) visual pos torch.Size([1, 36, 4])
logits shape torch.Size([7, 20, 30522])
 The criminal is arresting the cop. | -43.7295800447464

II torch.Size([11, 20]) AM torch.Size([11, 20])
visual features torch.Size([1, 36, 2048]) visual pos torch.Size([1, 36, 4])
logits shape torch.Size([11, 20, 30522])
 The babysitter is scolding the child. | -51.301841892302036

II torch.Size([11, 20]) AM torch.Size([11, 20])
visual features torch.Size([1, 36, 2048]) visual pos torch.Size([1, 36, 4])
logits shape torch.Size([11, 20, 30522])
 The child is scolding the babysitter. | -50.23145787976682

II torch.Size([13, 20]) AM torch.Size([13, 20])
visual features torch.Size([1, 36, 2048]) visual pos torch.Size([1, 36, 4])
logits shape torch.Size([13, 20, 30522])
 The doctor is using a stethoscope on the patient. | -51.90620003268123

II torch.Size([13, 20]) AM torch.Size([13, 20])
visual features torch.Size([1, 36, 2048]) 

#ViLT con immagini e testo

Le immagini del dataset EventsRev non vengono lette correttamente dal modello ViLT, infatti per tutti i logits la predizione è uguale a 1.

##Pseudo-log-likelihood

In [17]:
import torch

In [21]:
class Pseudo_log_likelihood:

    def __init__(self, tokenizer, model: torch.nn.Module, model_name, sentence, image):
        self.tokenizer = tokenizer
        self.model = model
        self.model_name = model_name
        self.sentence = sentence
        self.image = image
        self.cached_plls: Optional[np.ndarray] = None
        self.sent, self.score = self.pseudo_log_likelihood(self.sentence, self.model_name, self.image)


    def get_masked_seq(self, tokens, tokenized_words, tokenizer, model_name):
        """
        Masking out sentences word by word. If a word is tokenized into multiple subtokens, mask out subtokens linearly.
        """
        nr_masks = [len(elm) for elm in tokenized_words]

        masked_seq = []
        i = 1 #start with 0 because we want to predict for the first element
        j = 1
        while i < len(tokens) -1: #don't subtract 1 from len(tokens) to include the last element
            curr_masked_seq = []
            if nr_masks[j] == 1:
                curr_masked_seq = [tokens[ind] if ind != i else tokenizer.mask_token for ind in range(len(tokens))]
                masked_seq.append(curr_masked_seq)
                i += 1
                j += 1

            else:
                for k in range(nr_masks[j]):
                    curr_nr_masks = nr_masks[j] - k
                    curr_masked_seq = [tokens[ind] if (ind < i+k or ind >= i+k+curr_nr_masks) else tokenizer.mask_token for ind in range(len(tokens))]
                    masked_seq.append(curr_masked_seq)
                i += nr_masks[j]
                j += 1

        return masked_seq

    def prepare_input(self, sentence, model_name):

        tokens = self.tokenizer.tokenize(sentence)
        tokens = [self.tokenizer.cls_token] + tokens + [self.tokenizer.sep_token]
        #print(f"Tokens: {tokens}")
        tokenized_words = []

        for ind, tok in enumerate(tokens):
            if not re.match("#", tok):
                curr_word = [tok]
            else:
                curr_word.append(tok)

            #end a word
            if ind == len(tokens) - 1:
                tokenized_words.append(curr_word)
            else:
                if not re.match("#", tokens[ind+1]):
                    tokenized_words.append(curr_word)

        #print(f"Tokenized words: {tokenized_words}")

        masked_seq = self.get_masked_seq(tokens, tokenized_words, self.tokenizer, self.model_name)
        #print(f"Masked seq: {masked_seq}")

        return tokens, masked_seq

    def pseudo_log_likelihood(self, sentence, model_name, image):
        mask_token_id = self.tokenizer.mask_token_id
        max_len = 20

        tokens, masked_seq = self.prepare_input(sentence, model_name)
        nr_tokens_to_predict = len(tokens) - 2 #because of CLS & SEP

        """Per avere la stessa forma per input testuale e visuale
        ho ripetuto l'immagine tante volte quanto è la quantità di token da predire """
        list_of_sents = [sentence] * nr_tokens_to_predict
        list_of_images = [image] * nr_tokens_to_predict
        #print(list_of_images)
        processor = ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm")
        encoded_inputs = processor(list_of_images, list_of_sents, return_tensors = "pt")
        #print(encoded_inputs)
        for i in range(len(masked_seq)):
            for j in range(len(masked_seq[i])):
                if masked_seq[i][j] == self.tokenizer.mask_token:
                    encoded_inputs["input_ids"][i][j] = self.tokenizer.mask_token_id

        input_ids = torch.tensor(encoded_inputs["input_ids"])
        attention_mask = torch.tensor(encoded_inputs["attention_mask"])
        pixel_values = torch.tensor(encoded_inputs["pixel_values"])

        #II torch.Size([7, 20]) AM torch.Size([7, 20]) TTI torch.Size([7, 20])
        print("II", input_ids.shape, "AM", attention_mask.shape, "PV", pixel_values.shape)

        # calcola la pseudo-log-likelihood
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values = pixel_values,
                )
        log_probs_fillers = []
        all_log_probs=[]
        predict_token = tokens[1:-1]
        logits = outputs.logits
        print("shape of logits", logits.shape)

        for batch_elem, token, index in zip(range(len(logits)), predict_token, range(1, len(predict_token) + 1)): #no need for CLS & SEP
            all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])
            log_probs_fillers.append(all_log_probs[self.tokenizer.convert_tokens_to_ids(token)].item())
            #print(f"{self.tokenizer.convert_tokens_to_ids(token)} | {token} | {all_log_probs[self.tokenizer.convert_tokens_to_ids(token)].item()}\n")

        sentence_score = sum(log_probs_fillers)
        print(f" {sentence} | {sentence_score}\n")
        #self.cached_plls = np.array([sentence_score])

        return sentence, sentence_score

##main

In [22]:
import os
import torch
from transformers import ViltProcessor, ViltForMaskedLM
import pandas as pd
from PIL import Image
import cv2
import pickle
import glob
import json
import re

In [24]:
#from pseudo_log_likelihood import Pseudo_log_likelihood

def main():

    force = True
    #sentence_pll = True
    result_plls = {}
    img_w = 384
    img_h = 512
    print(f"\nforce={force}\n")
    if os.path.exists("/content/"):
        data_dir = "/content/drive/MyDrive/Data"
    else:
        data_dir = "data"
    img_folder= "/content/drive/MyDrive/Data/EventsRev_picture_stimuli/"
    models = {
        "ViLT": (
            ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm").tokenizer,
            ViltForMaskedLM.from_pretrained("dandelin/vilt-b32-mlm")
        ),
    }
    filename = (f"{data_dir}/eventsRev_demo.csv")
    print(filename)
    global_results = {}
    for model_name, (tokenizer, model) in list(models.items()):
        results = {}
        global_results[filename] = results
        print("File Name:", filename.split("/")[-1])
        out_file_name = os.path.join(
            "/content/drive/MyDrive/Data/results", f"{model_name}_{os.path.basename(filename)}_sentence.txt"
        )
        data = pd.read_csv(filename, sep=";")
        results["len(data)"] = len(data)
        """save the results with the file name"""
        if force or (not os.path.exists(f"{filename}_result_plls.pkl")):
            for model_name, (tokenizer, model) in list(models.items()):
                model_results = {}
                results[model_name] = model_results
                result = {
                        "sentences": [],
                        "sent_ppls": [],
                        "num_words": [],
                        "num_tokens": [],
                        }
                print(f"\n\nEvaluating: {model_name}")
                for idx, row in enumerate(data.itertuples()):
                    sentence = row.Sentence
                    image_filename = row.file_name
                    image_path = os.path.join(img_folder, image_filename)
                    image = cv2.imread(image_path)
                    image = cv2.cvtColor(image, cv2.IMREAD_COLOR)
                    resized_image = cv2.resize(image, (img_h, img_w))
                    pll = Pseudo_log_likelihood(
                            tokenizer, model, model_name, sentence, resized_image
                            )
                    tokens = tokenizer.tokenize(sentence)
                    num_tokens = len(tokens)
                    words = sentence.split()
                    num_words = len(words)
                    result["sentences"].append(pll.sent)
                    result["sent_ppls"].append(pll.score)
                    result["num_words"].append(num_words)
                    result["num_tokens"].append(num_tokens)

                # uses pandas to save the results in a csv file
                pd.DataFrame(result).to_csv(
                    out_file_name, sep="\t", header=None, index=None
                    )


if __name__ == '__main__':
    main()


force=True

/content/drive/MyDrive/Data/eventsRev_demo.csv
File Name: eventsRev_demo.csv


Evaluating: ViLT


<ipython-input-21-11e8ccbbb621>:91: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = torch.tensor(encoded_inputs["input_ids"])
<ipython-input-21-11e8ccbbb621>:92: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(encoded_inputs["attention_mask"])
<ipython-input-21-11e8ccbbb621>:93: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pixel_values = torch.tensor(encoded_inputs["pixel_values"])


II torch.Size([7, 9]) AM torch.Size([7, 9]) PV torch.Size([7, 3, 384, 512])
shape of logits torch.Size([7, 9, 30522])
 The cop is arresting the criminal. | -31.77325602993369



<ipython-input-21-11e8ccbbb621>:113: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])


II torch.Size([7, 9]) AM torch.Size([7, 9]) PV torch.Size([7, 3, 384, 512])
shape of logits torch.Size([7, 9, 30522])
 The criminal is arresting the cop. | -30.38425577059388

II torch.Size([11, 13]) AM torch.Size([11, 13]) PV torch.Size([11, 3, 384, 512])
shape of logits torch.Size([11, 13, 30522])
 The babysitter is scolding the child. | -50.120992325246334

II torch.Size([11, 13]) AM torch.Size([11, 13]) PV torch.Size([11, 3, 384, 512])
shape of logits torch.Size([11, 13, 30522])
 The child is scolding the babysitter. | -50.22989095747471

II torch.Size([13, 15]) AM torch.Size([13, 15]) PV torch.Size([13, 3, 384, 512])
shape of logits torch.Size([13, 15, 30522])
 The doctor is using a stethoscope on the patient. | -8.29768533189781

II torch.Size([13, 15]) AM torch.Size([13, 15]) PV torch.Size([13, 3, 384, 512])
shape of logits torch.Size([13, 15, 30522])
 The patient is using a stethoscope on the doctor. | -15.853064490482211



#FLAVA con immagini e testo

##pseudo-log-likelihood

In [25]:
import torch
from transformers import FlavaForPreTraining, AutoProcessor

In [26]:

class Pseudo_log_likelihood:

    def __init__(self, tokenizer, model: torch.nn.Module, model_name, sentence, image):
        self.tokenizer = tokenizer
        self.model = model
        self.model_name = model_name
        self.sentence = sentence
        self.image = image
        self.cached_plls: Optional[np.ndarray] = None
        self.sent, self.score = self.pseudo_log_likelihood(self.sentence, self.model_name, self.image)


    def get_masked_seq(self, tokens, tokenized_words, tokenizer, model_name):
        """
        Masking out sentences word by word. If a word is tokenized into multiple subtokens, mask out subtokens linearly.
        """
        nr_masks = [len(elm) for elm in tokenized_words]

        masked_seq = []
        i = 1 #start with 0 because we want to predict for the first element
        j = 1
        while i < len(tokens) -1: #don't subtract 1 from len(tokens) to include the last element
            curr_masked_seq = []
            if nr_masks[j] == 1:
                curr_masked_seq = [tokens[ind] if ind != i else tokenizer.mask_token for ind in range(len(tokens))]
                masked_seq.append(curr_masked_seq)
                i += 1
                j += 1

            else:
                for k in range(nr_masks[j]):
                    curr_nr_masks = nr_masks[j] - k
                    curr_masked_seq = [tokens[ind] if (ind < i+k or ind >= i+k+curr_nr_masks) else tokenizer.mask_token for ind in range(len(tokens))]
                    masked_seq.append(curr_masked_seq)
                i += nr_masks[j]
                j += 1

        return masked_seq

    def prepare_input(self, sentence, model_name):

        tokens = self.tokenizer.tokenize(sentence)
        tokens = [self.tokenizer.cls_token] + tokens + [self.tokenizer.sep_token]
        #print(f"Tokens: {tokens}")
        tokenized_words = []
        for ind, tok in enumerate(tokens):
            if not re.match("#", tok):
                curr_word = [tok]
            else:
                curr_word.append(tok)

            #end a word
            if ind == len(tokens) - 1:
                tokenized_words.append(curr_word)
            else:
                if not re.match("#", tokens[ind+1]):
                    tokenized_words.append(curr_word)

        #print(f"Tokenized words: {tokenized_words}")

        masked_seq = self.get_masked_seq(tokens, tokenized_words, self.tokenizer, self.model_name)
        #print(f"Masked seq: {masked_seq}")

        return tokens, masked_seq

    def pseudo_log_likelihood(self, sentence, model_name, image):
        #print(sentence)
        mask_token_id = self.tokenizer.mask_token_id
        max_len = 20

        tokens, masked_seq = self.prepare_input(sentence, model_name)
        nr_tokens_to_predict = len(tokens) - 2 #because of CLS & SEP

        #copio l'immagine tante volte quanto è il numero di token della frase
        list_of_sents = [sentence] * nr_tokens_to_predict
        list_of_images = [image] * nr_tokens_to_predict
        processor = AutoProcessor.from_pretrained("facebook/flava-full")
        encoded_inputs = processor(images=list_of_images,
                                   text=list_of_sents,
                                   padding=True,
                                   max_length=77,
                                   return_tensors="pt")

        for i in range(len(masked_seq)):
            for j in range(len(masked_seq[i])):
                if masked_seq[i][j] == self.tokenizer.mask_token:
                    encoded_inputs["input_ids"][i][j] = self.tokenizer.mask_token_id

        ## per calcolare la predizione della maschera utilizzo input_ids_masked
        input_ids_masked = torch.tensor(encoded_inputs["input_ids"])
        attention_mask = torch.tensor(encoded_inputs["attention_mask"])
        pixel_values = torch.tensor(encoded_inputs["pixel_values"])

        print("Input Ids Masked", input_ids_masked.shape, "Attention Mask", attention_mask.shape, "Pixel Values", pixel_values.shape)

        # calcola la pseudo-log-likelihood
        self.model.eval()
        with torch.no_grad():
            outputs = self.model(
                input_ids_masked = input_ids_masked,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                return_loss = False
                )
        log_probs_fillers = []
        all_log_probs=[]
        predict_token = tokens[1:-1]

        logits = outputs.mmm_text_logits
        print("Shape of logits:", logits.shape)


        for batch_elem, token, index in zip(range(len(logits)), predict_token, range(1, len(predict_token) + 1)): #no need for CLS & SEP
            all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])
            log_probs_fillers.append(all_log_probs[self.tokenizer.convert_tokens_to_ids(token)].item())
            #print(f"{self.tokenizer.convert_tokens_to_ids(token)} | {token} | {all_log_probs[self.tokenizer.convert_tokens_to_ids(token)].item()}\n")

        sentence_score = sum(log_probs_fillers)
        print(f" {sentence} | {sentence_score}\n")

        return sentence, sentence_score

##main

In [27]:
import os
import torch
import pandas as pd
from PIL import Image
import cv2
import pickle
import glob
import json
import re

In [29]:

def main():

    force = True
    result_plls = {}

    ##dimensioni di defalut di altezza e larghezza dell'immagine
    img_w = 224
    img_h = 224

    print(f"\nforce={force}\n")
    if os.path.exists("/content/"):
        data_dir = "/content/drive/MyDrive/Data"
    else:
        data_dir = "data"
    img_folder= "/content/drive/MyDrive/Data/EventsRev_picture_stimuli/"

    models = {
        "FLAVA": (
            AutoProcessor.from_pretrained("facebook/flava-full").tokenizer,
            FlavaForPreTraining.from_pretrained("facebook/flava-full")
        ),
    }
    filename = (f"{data_dir}/eventsRev_demo.csv")
    print(filename)
    global_results = {}
    for model_name, (tokenizer, model) in list(models.items()):
        results = {}
        global_results[filename] = results
        print("File Name:", filename.split("/")[-1])
        out_file_name = os.path.join(
            "/content/drive/MyDrive/Data/results", f"{model_name}_{os.path.basename(filename)}_sentence.txt"
        )
        data = pd.read_csv(filename, sep=";")
        results["len(data)"] = len(data)

        """save the results with the file name"""
        if force or (not os.path.exists(f"{filename}_result_plls.pkl")):
            for model_name, (tokenizer, model) in list(models.items()):
                model_results = {}
                results[model_name] = model_results
                result = {
                        "sentences": [],
                        "sent_ppls": [],
                        "num_words": [],
                        "num_tokens": [],
                        }
                print(f"\n\nEvaluating: {model_name}")

                """Leggo l'immagine sulla base del nome del file corrispodente a ciascuna frase"""
                for idx, row in enumerate(data.itertuples()):
                    sentence = row.Sentence
                    image_filename = row.file_name
                    image_path = os.path.join(img_folder, image_filename)
                    image = cv2.imread(image_path)
                    image = cv2.cvtColor(image, cv2.IMREAD_COLOR)
                    ##applico il ridimensionamento all'immagine
                    resized_image = cv2.resize(image, (img_h, img_w))

                    pll = Pseudo_log_likelihood(
                            tokenizer, model, model_name, sentence, resized_image
                            )

                    tokens = tokenizer.tokenize(sentence)
                    num_tokens = len(tokens)
                    words = sentence.split()
                    num_words = len(words)
                    result["sentences"].append(pll.sent)
                    result["sent_ppls"].append(pll.score)
                    result["num_words"].append(num_words)
                    result["num_tokens"].append(num_tokens)

                # salvo i risultati
                pd.DataFrame(result).to_csv(
                    out_file_name, sep="\t", header=None, index=None
                    )


if __name__ == '__main__':
    main()



force=True



`text_config_dict` is provided which will be used to initialize `FlavaTextConfig`. The value `text_config["id2label"]` will be overriden.
`multimodal_config_dict` is provided which will be used to initialize `FlavaMultimodalConfig`. The value `multimodal_config["id2label"]` will be overriden.
`image_codebook_config_dict` is provided which will be used to initialize `FlavaImageCodebookConfig`. The value `image_codebook_config["id2label"]` will be overriden.


/content/drive/MyDrive/Data/eventsRev_demo.csv
File Name: eventsRev_demo.csv


Evaluating: FLAVA


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2395: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
<ipython-input-26-e954a3f60bc5>:94: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids_masked = torch.tensor(encoded_inputs["input_ids"])
<ipython-input-26-e954a3f60bc5>:95: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  attention_mask = torch.tensor(encoded_inputs["attention_mask"])
<ipython-input-26-e954a3f60bc5>:96: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone

Input Ids Masked torch.Size([7, 9]) Attention Mask torch.Size([7, 9]) Pixel Values torch.Size([7, 3, 224, 224])


/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:884: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Shape of logits: torch.Size([7, 9, 30522])
 The cop is arresting the criminal. | -12.916678346693516



<ipython-input-26-e954a3f60bc5>:119: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])


Input Ids Masked torch.Size([7, 9]) Attention Mask torch.Size([7, 9]) Pixel Values torch.Size([7, 3, 224, 224])
Shape of logits: torch.Size([7, 9, 30522])
 The criminal is arresting the cop. | -22.69902354478836

Input Ids Masked torch.Size([11, 13]) Attention Mask torch.Size([11, 13]) Pixel Values torch.Size([11, 3, 224, 224])
Shape of logits: torch.Size([11, 13, 30522])
 The babysitter is scolding the child. | -33.85025492310524

Input Ids Masked torch.Size([11, 13]) Attention Mask torch.Size([11, 13]) Pixel Values torch.Size([11, 3, 224, 224])
Shape of logits: torch.Size([11, 13, 30522])
 The child is scolding the babysitter. | -41.23013115674257

Input Ids Masked torch.Size([13, 15]) Attention Mask torch.Size([13, 15]) Pixel Values torch.Size([13, 3, 224, 224])
Shape of logits: torch.Size([13, 15, 30522])
 The doctor is using a stethoscope on the patient. | -40.2151440307498

Input Ids Masked torch.Size([13, 15]) Attention Mask torch.Size([13, 15]) Pixel Values torch.Size([13, 3, 2